In [ ]:
# CodeCrossEnc-v1: Train + Eval
# Base: cross-encoder/ms-marco-MiniLM-L-6-v2
# Data: CodeSearchNet python train, 100K pos + 2 rand neg each = 300K pairs
# 1 epoch, batch=32, lr=2e-5, warmup=500, max_length=384, T4 ~75min
#
# FUSE sync fix zorunlu: save_model -> sync -> sleep(120) -> ls dogrulama
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CROSS_PATH = '/content/drive/MyDrive/adaptmem-bench/code-crossenc/v1'
BI_PATH    = '/content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-5000/model'
OUT_DIR    = '/content/drive/MyDrive/adaptmem-bench/eval/codecrossenc-v1'

import os
os.makedirs(OUT_DIR, exist_ok=True)
print('Drive mounted. BI_PATH:', BI_PATH)
print('CROSS_PATH will be:', CROSS_PATH)

In [ ]:
!pip install -q sentence-transformers datasets

In [ ]:
# ---- TRAIN CodeCrossEnc-v1 ----
import time, random
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

print('[train] loading CodeSearchNet python train split...')
ds_train = load_dataset('code_search_net', 'python', split='train')

MAX_PAIRS = 100_000
N_NEG = 2
random.seed(42)

all_codes = [r['func_code_string'] for r in ds_train if len(r.get('func_code_string','')) >= 40]
samples = []
for r in ds_train:
    doc = (r.get('func_documentation_string') or '').strip()
    code = (r.get('func_code_string') or '').strip()
    if len(doc) < 10 or len(code) < 40:
        continue
    q = doc.splitlines()[0][:200]
    samples.append((q, code))
    if len(samples) >= MAX_PAIRS:
        break

train_examples = []
for q, pos in samples:
    train_examples.append(InputExample(texts=[q, pos], label=1.0))
    for _ in range(N_NEG):
        neg = random.choice(all_codes)
        train_examples.append(InputExample(texts=[q, neg], label=0.0))

print(f'[train] {len(train_examples)} pairs ({len(samples)} pos + {len(samples)*N_NEG} neg)')
random.shuffle(train_examples)

cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
loader = DataLoader(train_examples, shuffle=True, batch_size=32)

t0 = time.time()
cross.fit(
    train_dataloader=loader,
    epochs=1,
    warmup_steps=500,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True,
)
print(f'[train] done in {(time.time()-t0)/60:.1f}min')

In [ ]:
# ---- FUSE SYNC FIX: save -> sync -> sleep(120) -> dogrula ----
import subprocess, time as _time

print(f'[save] saving to {CROSS_PATH} ...')
cross.save(CROSS_PATH)
print('[save] save_model() returned, running sync...')
subprocess.run(['sync'])
print('[save] sync done, sleeping 120s for FUSE upload buffer...')
_time.sleep(120)
import os
files = os.listdir(CROSS_PATH)
print(f'[save] Drive dogrulama - {CROSS_PATH}:', files)
assert len(files) > 0, 'HATA: Drive bos, FUSE sync basarisiz!'
print('[save] OK, model Drive da.')

In [ ]:
# ---- EVAL: FT-Code-5000 bi-encoder top-20 -> CodeCrossEnc-v1 rerank ----
import numpy as np, json, time
from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder

def load_test(n=-1):
    ds = load_dataset('code_search_net', 'python', split='test')
    queries, truths, corpus_ids, corpus_texts = [], [], [], []
    seen = set()
    body_map = {}
    for i, row in enumerate(ds):
        if n > 0 and len(queries) >= n:
            break
        body = row.get('func_code_string') or ''
        doc  = (row.get('func_documentation_string') or '').strip()
        if len(body) < 40 or len(doc) < 10:
            continue
        key = body[:200]
        if key in seen:
            continue
        seen.add(key)
        cid = f'ts{i}'
        queries.append(doc.splitlines()[0][:200])
        truths.append(cid)
        corpus_ids.append(cid)
        corpus_texts.append(body)
    return queries, truths, corpus_ids, corpus_texts

t0 = time.time()
queries, truths, corpus_ids, corpus_texts = load_test(n=-1)
n_q = len(queries)
print(f'[load] {n_q} queries, {len(corpus_ids)} corpus in {time.time()-t0:.1f}s')

TOP_K = 20
print('[bi] loading FT-Code-5000...')
bi = SentenceTransformer(BI_PATH)
bi.max_seq_length = 256

t1 = time.time()
corpus_emb = bi.encode(corpus_texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
query_emb  = bi.encode(queries,      batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
sims = query_emb @ corpus_emb.T
topk_u = np.argpartition(-sims, TOP_K, axis=1)[:, :TOP_K]
rows   = np.arange(n_q)[:, None]
ts     = sims[rows, topk_u]
order  = np.argsort(-ts, axis=1)
topk   = topk_u[rows, order]
print(f'[bi] encoded+retrieved in {time.time()-t1:.1f}s')

# Bi-alone sanity
cid2idx = {c: i for i, c in enumerate(corpus_ids)}
tarr    = np.array([cid2idx[t] for t in truths])
bi_r1=bi_r5=bi_r10=0; bi_mrr=0.0
for qi in range(n_q):
    pos = np.where(topk[qi] == tarr[qi])[0]
    if not len(pos): continue
    p = int(pos[0])
    if p==0: bi_r1+=1
    if p<5:  bi_r5+=1
    if p<10: bi_r10+=1
    bi_mrr += 1/(p+1)
print(f'[bi-alone] R@1={bi_r1/n_q:.4f} R@5={bi_r5/n_q:.4f} R@10={bi_r10/n_q:.4f} MRR={bi_mrr/n_q:.4f}')

print('[cross] loading CodeCrossEnc-v1...')
cross = CrossEncoder(CROSS_PATH, max_length=384)

pairs = [(queries[qi], corpus_texts[int(c)]) for qi in range(n_q) for c in topk[qi]]
print(f'[cross] scoring {len(pairs)} pairs...')
CHUNK = 4096
scores = np.zeros(len(pairs), dtype=np.float32)
for s in range(0, len(pairs), CHUNK):
    e = min(s+CHUNK, len(pairs))
    scores[s:e] = cross.predict(pairs[s:e], batch_size=64, show_progress_bar=(s==0))
scores = scores.reshape(n_q, TOP_K)
ro     = np.argsort(-scores, axis=1)
rerank = topk[np.arange(n_q)[:, None], ro]

r1=r5=r10=0; mrr=0.0
for qi in range(n_q):
    pos = np.where(rerank[qi] == tarr[qi])[0]
    if not len(pos): continue
    p = int(pos[0])+1
    if p==1:  r1+=1
    if p<=5:  r5+=1
    if p<=10: r10+=1
    mrr += 1/p

summary = {
    'bi_alone':  {'R@1':bi_r1/n_q,'R@5':bi_r5/n_q,'R@10':bi_r10/n_q,'MRR':bi_mrr/n_q},
    'reranked':  {'R@1':r1/n_q,   'R@5':r5/n_q,   'R@10':r10/n_q,   'MRR':mrr/n_q},
    'delta_R@1': (r1-bi_r1)/n_q,
    'delta_MRR': (mrr-bi_mrr)/n_q,
    'n_queries': n_q, 'top_k': TOP_K,
}
(Path(OUT_DIR)/'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

gate = summary['reranked']['R@1']
if gate >= 0.93:
    print(f'GATE PASS: R@1={gate:.4f} >= 0.93. Encoder+reranker axes compose!')
else:
    print(f'GATE FAIL: R@1={gate:.4f} < 0.93. Hard negative mining v2 lazim.')